In [18]:
# Importy
import os, json, pickle, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
#from tqdm.notebook import tqdm 

from functions import DoomDataset

In [27]:
# Parametry
VOCAB_SIZE = 579
EMBEDDING_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 3
STRIDE = 64
CONTEXT = 256
BATCH_SIZE = 128
EPOCHS = 25

os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)

PROC_DIR = '../data/processed/'
MODEL_DIR = '../models/'

In [20]:
# Ustwaianie ziarna i cuda
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [21]:
# Ładowanie danych i słownika tokenów
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = data.values()
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 43 | Vocab: 579 | Tokens: 384,327


In [22]:
# Przygotowanie DataLoadera
sequences_list = list(sequences)
train_dataset = DoomDataset(sequences_list, context = CONTEXT, stride = STRIDE, augment=False)
print(f"Total training samples generated: {len(train_dataset)}")
print(f"With batch_size={BATCH_SIZE}, one epoch will take {len(train_dataset) // BATCH_SIZE} steps.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

Total training samples generated: 5852
With batch_size=128, one epoch will take 45 steps.


In [23]:
# Pobranie jednej paczki danych
features, labels = next(iter(train_loader))

# Sprawdzenie kształtu danych
print(f"Kształt cech (X): {features.shape}")
print(f"Kształt etykiet (y): {labels.shape}")


Kształt cech (X): torch.Size([128, 256])
Kształt etykiet (y): torch.Size([128, 256])


In [24]:
# Definicja modelu
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(MusicLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.2)
        
        self.ln = nn.LayerNorm(hidden_size)

        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        
        embedded = self.embedding(x)
        
        out, hidden = self.lstm(embedded, hidden)
        
       
        out = self.ln(out)
        logits = self.fc(out)
        
        return logits, hidden

In [25]:
# Inicjalizacja modelu, optymalizatora i funkcji straty
model = MusicLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)


In [ ]:
# Trenowanie modelu + progress bar
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    total_batches = len(train_loader)
    
    # progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()

        logits, _ = model(inputs)

        loss = loss_fn(logits.view(-1, VOCAB_SIZE), targets.view(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
        current_loss = loss.item()

        #progress_bar.set_postfix({
        #        'loss': f"{current_loss:.4f}",
        #        'avg': f"{total_loss / (batch_idx +1):.4f}"
        #    })
        if batch_idx % 50 == 0 or batch_idx == len(train_loader) - 1:
            current_avg_loss = total_loss / (batch_idx +1)
            print(f"Epoch: [{epoch+1}/{EPOCHS}] | Batch: [{batch_idx + 1}/{total_batches}] | Current Loss: {current_avg_loss:.4f}")

    epoch_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} summary: avg loss = {epoch_loss:.4f}\n")

NameError: name 'total_batches' is not defined

In [ ]:
# Zapis modelu
torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'lstm_music_model.pth'))